<a href="https://colab.research.google.com/github/csugasi/DATASCI266-Banking77-Encoder-vs-Decoder/blob/shuchi%2Fdev/notebooks/qwen_finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
from google.colab import userdata
import os

token = userdata.get('GH_TOKEN')
repo  = "csugasi/DATASCI266-Banking77-Encoder-vs-Decoder"
path  = "/content/repo"

if not os.path.exists(path):
    !git clone -q -b shuchi/dev https://{token}@github.com/{repo}.git {path}

%cd {path}
!nvidia-smi

/content/repo
Mon Nov 16 17:20:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P0             49W /  400W |   17622MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+---------------------------------

In [16]:
!ls results/ | grep qwen

In [18]:
import pandas as pd, json
from datasets import Dataset

train = pd.read_parquet("data/splits/full_train.parquet")
val   = pd.read_parquet("data/splits/val.parquet")
labels = json.load(open("data/raw/label_names.json"))

print("train:", train.shape, "| val:", val.shape, "| labels:", len(labels))

train: (9618, 3) | val: (385, 3) | labels: 77


In [19]:
import time
import numpy as np
from datasets import Dataset
from trl import SFTConfig, SFTTrainer

In [20]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.bfloat16, device_map="cuda"
)

print("Model loaded — parameters:", f"{sum(p.numel() for p in model.parameters()):,}")
print("GPU memory used:", round(torch.cuda.memory_allocated()/1e9, 2), "GB")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model loaded — parameters: 1,543,714,304
GPU memory used: 15.46 GB


In [21]:
# same chat format we used for zero-shot, except fine tuning now

def format_example(row):
    """Turn one row into a chat the model learns to complete."""
    intent_name = labels[row["label"]]
    return {
        "messages": [
            {"role": "system", "content": "You classify banking customer messages. "
             "Reply with exactly one intent name and nothing else."},
            {"role": "user", "content": row["text"]},
            {"role": "assistant", "content": intent_name},
        ]
    }

# Format all training examples
train_chats = [format_example(row) for _, row in train.iterrows()]
val_chats   = [format_example(row) for _, row in val.iterrows()]

print("Training examples:", len(train_chats))
print("First example:")
for msg in train_chats[0]["messages"]:
    print(f"  {msg['role']}: {msg['content']}")

Training examples: 9618
First example:
  system: You classify banking customer messages. Reply with exactly one intent name and nothing else.
  user: I am still waiting on my card?
  assistant: card_arrival


In [22]:
!pip install -q trl

In [ ]:
# set up time traner to run one epoch
import time
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

train_ds = Dataset.from_list(train_chats)

config = SFTConfig(
    output_dir="./timing_test",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    bf16=True,
    logging_steps=50,
    save_strategy="no",
    report_to="none",
    max_length=256,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_ds,
    args=config,
)

print("Setup done. Starting training...")
print(f"GPU memory before training: {torch.cuda.memory_allocated()/1e9:.2f} GB")

start = time.time()
trainer.train()
elapsed = time.time() - start

print(f"\n--- TIMING RESULTS ---")
print(f"One epoch took: {elapsed/60:.1f} minutes")
print(f"GPU memory peak: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")
print(f"GPU memory now:  {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# Find the best learning rate

import numpy as np

# Load k5 seed0 training data
k5 = pd.read_parquet("data/splits/train_k5_seed0.parquet")
k5_chats = [format_example(row) for _, row in k5.iterrows()]
k5_ds = Dataset.from_list(k5_chats)

print("k5 training examples:", len(k5_chats))

# Prepare validation data for evaluation
val_texts = val["text"].tolist()
val_labels = val["label"].to_numpy()

# Test 3 learning rates
for lr in [2e-5, 5e-5, 1e-4]:
    # Reload fresh model each time
    fresh_model = AutoModelForCausalLM.from_pretrained(
        MODEL, dtype=torch.bfloat16, device_map="cuda" )

    cfg = SFTConfig(
        output_dir=f"./lr_test_{lr}",
        num_train_epochs=3,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        learning_rate=lr,
        bf16=True,
        logging_steps=20,
        save_strategy="no",
        report_to="none",
        max_length=256,)

    t = SFTTrainer(
        model=fresh_model,
        processing_class=tokenizer,
        train_dataset=k5_ds,
        args=cfg,)

    print(f"\n--- LR = {lr} ---")
    t.train()
    print(f"Final loss: {t.state.log_history[-1].get('train_loss', 'N/A')}")

    # Clean up GPU memory
    del fresh_model, t
    torch.cuda.empty_cache()

print("\nDone! Check which LR had the lowest final loss.")

Final choice to go with e-5 since it’s a good middle ground - close to the best loss, and less likely to blow up on tiny datasets.

In [23]:
# Define classify again in this notebook

def build_prompt(message):
    chat = [
        {"role": "system", "content": "You classify banking customer messages. Reply with exactly one intent "
         "name from the list, copied exactly, and nothing else."},
        {"role": "user", "content": f"Intents:\n{INTENT_LIST}\n\nMessage: {message}\n\nIntent:"},
    ]
    return tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)

INTENT_LIST = "\n".join(labels)

def classify(messages, batch_size=16):
    outputs = []
    for i in range(0, len(messages), batch_size):
        prompts = [build_prompt(m) for m in messages[i:i + batch_size]]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True).to("cuda")
        with torch.no_grad():
            generated = model.generate(**inputs, max_new_tokens=20, do_sample=False)
        new_tokens = generated[:, inputs["input_ids"].shape[1]:]
        outputs += tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
    return outputs

val_texts = val["text"].tolist()
val_labels = val["label"].to_numpy()

print("classify function ready")

classify function ready


In [24]:
# Real Training, All Budgets, All seeds

import sys
sys.path.insert(0, "scripts")
from predictions import match_intent, write_predictions

# All budget files
import glob
budget_files = sorted(glob.glob("data/splits/train_*.parquet"))

LEARNING_RATE = 5e-5
EPOCHS = 3
results = []

for bf in budget_files:
    # Extract budget and seed from filename: train_k5_seed0.parquet
    parts = bf.split("/")[-1].replace("train_", "").replace(".parquet", "").split("_")
    budget = parts[0]       # e.g. "k5"
    seed = parts[1]         # e.g. "seed0"
    seed_num = int(seed.replace("seed", ""))

    print(f"\n{'='*50}")
    print(f"Training: budget={budget}, {seed}")
    print(f"{'='*50}")

    # Load training data
    tr = pd.read_parquet(bf)
    tr_chats = [format_example(row) for _, row in tr.iterrows()]
    tr_ds = Dataset.from_list(tr_chats)
    print(f"Training examples: {len(tr_chats)}")

    # Fresh model
    fresh_model = AutoModelForCausalLM.from_pretrained(
        MODEL, dtype=torch.bfloat16, device_map="cuda"
    )

    cfg = SFTConfig(
        output_dir=f"./runs/{budget}_{seed}",
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        learning_rate=LEARNING_RATE,
        bf16=True,
        logging_steps=20,
        save_strategy="no",
        report_to="none",
        max_length=256,
    )

    t = SFTTrainer(
        model=fresh_model,
        processing_class=tokenizer,
        train_dataset=tr_ds,
        args=cfg,
    )

    start = time.time()
    t.train()
    elapsed = time.time() - start
    print(f"Training took: {elapsed/60:.1f} minutes")

    # Evaluate on validation set
    fresh_model.eval()
    model = fresh_model # so classify can use it
    raw = classify(val_texts)
    pred = np.array([match_intent(r) for r in raw])
    acc = (pred == val_labels).mean()
    print(f"Val accuracy: {acc:.3f}")

    # Save predictions
    path = write_predictions("qwen_qlora", budget, seed_num, "val", pred, raw_outputs=raw)
    print(f"Saved: {path}")

    results.append({"budget": budget, "seed": seed, "acc": acc, "time": elapsed/60})

    # Clean up
    del fresh_model, t
    torch.cuda.empty_cache()

# Summary
print(f"\n{'='*50}")
print("SUMMARY")
print(f"{'='*50}")
for r in results:
    print(f"{r['budget']} {r['seed']}: acc={r['acc']:.3f}, time={r['time']:.1f}min")


Training: budget=k10, seed0
Training examples: 770


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Tokenizing train dataset:   0%|          | 0/770 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/770 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/770 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/770 [00:00<?, ? examples/s]

... (139 outputs left out)

Truncating train dataset:   0%|          | 0/4807 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/4807 [00:00<?, ? examples/s]

Step,Training Loss
20,1.268129
40,0.752106
60,0.719234
80,0.705845
100,0.716314
120,0.686119
140,0.681680
160,0.649205
180,0.675087
200,0.620953


Training took: 14.1 minutes
Val accuracy: 0.784
Saved: /content/repo/results/qwen_qlora_p50_seed2_val.csv

SUMMARY
k10 seed0: acc=0.587, time=2.3min
k10 seed1: acc=0.595, time=2.3min
k10 seed2: acc=0.634, time=2.2min
k1 seed0: acc=0.416, time=0.2min
k1 seed1: acc=0.408, time=0.2min
k1 seed2: acc=0.413, time=0.2min
k20 seed0: acc=0.696, time=4.5min
k20 seed1: acc=0.766, time=4.5min
k20 seed2: acc=0.670, time=4.5min
k30 seed0: acc=0.782, time=6.8min
k30 seed1: acc=0.743, time=6.7min
k30 seed2: acc=0.704, time=6.8min
k3 seed0: acc=0.434, time=0.7min
k3 seed1: acc=0.455, time=0.7min
k3 seed2: acc=0.418, time=0.7min
k5 seed0: acc=0.494, time=1.1min
k5 seed1: acc=0.504, time=1.2min
k5 seed2: acc=0.481, time=1.1min
p50 seed0: acc=0.764, time=14.2min
p50 seed1: acc=0.784, time=14.2min
p50 seed2: acc=0.784, time=14.1min


In [26]:
!git config user.email "shuchivardan@gmail.com"
!git config user.name "Shuchi Vardan"
!git add results/
!git commit -m "Add Qwen SFT val predictions for all budgets and seeds"
!git push origin shuchi/dev

[shuchi/dev 4bedbe2] Add Qwen SFT val predictions for all budgets and seeds
 21 files changed, 8106 insertions(+)
 create mode 100644 results/qwen_qlora_k10_seed0_val.csv
 create mode 100644 results/qwen_qlora_k10_seed1_val.csv
 create mode 100644 results/qwen_qlora_k10_seed2_val.csv
 create mode 100644 results/qwen_qlora_k1_seed0_val.csv
 create mode 100644 results/qwen_qlora_k1_seed1_val.csv
 create mode 100644 results/qwen_qlora_k1_seed2_val.csv
 create mode 100644 results/qwen_qlora_k20_seed0_val.csv
 create mode 100644 results/qwen_qlora_k20_seed1_val.csv
 create mode 100644 results/qwen_qlora_k20_seed2_val.csv
 create mode 100644 results/qwen_qlora_k30_seed0_val.csv
 create mode 100644 results/qwen_qlora_k30_seed1_val.csv
 create mode 100644 results/qwen_qlora_k30_seed2_val.csv
 create mode 100644 results/qwen_qlora_k3_seed0_val.csv
 create mode 100644 results/qwen_qlora_k3_seed1_val.csv
 create mode 100644 results/qwen_qlora_k3_seed2_val.csv
 create mode 100644 results/qwen_qlor

In [27]:
# check test file location

import os
# Find the test file
for p in ["data/splits/test.parquet", "data/raw/test.parquet"]:
    if os.path.exists(p):
        print(f"Found: {p}")
        test_df = pd.read_parquet(p)
        print(f"Test examples: {test_df.shape}")
        break
else:
    print("No test file found! Run: !find . -name '*test*'")

Found: data/raw/test.parquet
Test examples: (3080, 2)


In [28]:
# test eval loop

# Test set evaluation — all budgets, all seeds

test_texts = test_df["text"].tolist()
test_labels = test_df["label"].to_numpy()

test_results = []

for bf in budget_files:
    parts = bf.split("/")[-1].replace("train_", "").replace(".parquet", "").split("_")
    budget = parts[0]
    seed = parts[1]
    seed_num = int(seed.replace("seed", ""))

    print(f"\n{'='*50}")
    print(f"Training: budget={budget}, {seed}")
    print(f"{'='*50}")

    tr = pd.read_parquet(bf)
    tr_chats = [format_example(row) for _, row in tr.iterrows()]
    tr_ds = Dataset.from_list(tr_chats)
    print(f"Training examples: {len(tr_chats)}")

    fresh_model = AutoModelForCausalLM.from_pretrained(
        MODEL, dtype=torch.bfloat16, device_map="cuda"
    )

    cfg = SFTConfig(
        output_dir=f"./runs_test/{budget}_{seed}",
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        learning_rate=LEARNING_RATE,
        bf16=True,
        logging_steps=20,
        save_strategy="no",
        report_to="none",
        max_length=256,
    )

    t = SFTTrainer(
        model=fresh_model,
        processing_class=tokenizer,
        train_dataset=tr_ds,
        args=cfg,
    )

    start = time.time()
    t.train()
    elapsed = time.time() - start
    print(f"Training took: {elapsed/60:.1f} minutes")

    fresh_model.eval()
    model = fresh_model
    raw = classify(test_texts)
    pred = np.array([match_intent(r) for r in raw])
    acc = (pred == test_labels).mean()
    print(f"Test accuracy: {acc:.3f}")

    path = write_predictions("qwen_qlora", budget, seed_num, "test", pred, raw_outputs=raw)
    print(f"Saved: {path}")

    test_results.append({"budget": budget, "seed": seed, "acc": acc, "time": elapsed/60})

    del fresh_model, t
    torch.cuda.empty_cache()

print(f"\n{'='*50}")
print("TEST SUMMARY")
print(f"{'='*50}")
for r in test_results:
    print(f"{r['budget']} {r['seed']}: acc={r['acc']:.3f}, time={r['time']:.1f}min")


Training: budget=k10, seed0
Training examples: 770


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Tokenizing train dataset:   0%|          | 0/770 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/770 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/770 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/770 [00:00<?, ? examples/s]

... (139 outputs left out)

Truncating train dataset:   0%|          | 0/4807 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/4807 [00:00<?, ? examples/s]

Step,Training Loss
20,1.268129
40,0.752106
60,0.719234
80,0.705845
100,0.716314
120,0.686119
140,0.681680
160,0.649205
180,0.675087
200,0.620953


Training took: 14.1 minutes
Test accuracy: 0.778
Saved: /content/repo/results/qwen_qlora_p50_seed2_test.csv

TEST SUMMARY
k10 seed0: acc=0.590, time=2.3min
k10 seed1: acc=0.624, time=2.3min
k10 seed2: acc=0.646, time=2.3min
k1 seed0: acc=0.467, time=0.2min
k1 seed1: acc=0.465, time=0.2min
k1 seed2: acc=0.457, time=0.2min
k20 seed0: acc=0.697, time=4.5min
k20 seed1: acc=0.770, time=4.5min
k20 seed2: acc=0.694, time=4.5min
k30 seed0: acc=0.785, time=6.9min
k30 seed1: acc=0.776, time=6.9min
k30 seed2: acc=0.724, time=6.8min
k3 seed0: acc=0.494, time=0.7min
k3 seed1: acc=0.501, time=0.7min
k3 seed2: acc=0.494, time=0.7min
k5 seed0: acc=0.520, time=1.1min
k5 seed1: acc=0.538, time=1.1min
k5 seed2: acc=0.533, time=1.1min
p50 seed0: acc=0.759, time=14.1min
p50 seed1: acc=0.800, time=14.2min
p50 seed2: acc=0.778, time=14.1min


In [29]:
# check all pred files
import glob
pred_files = sorted(glob.glob("results/qwen_qlora_*.csv"))
print(f"Total prediction files: {len(pred_files)}")
for f in pred_files:
    print(f"  {f}")

Total prediction files: 42
  results/qwen_qlora_k10_seed0_test.csv
  results/qwen_qlora_k10_seed0_val.csv
  results/qwen_qlora_k10_seed1_test.csv
  results/qwen_qlora_k10_seed1_val.csv
  results/qwen_qlora_k10_seed2_test.csv
  results/qwen_qlora_k10_seed2_val.csv
  results/qwen_qlora_k1_seed0_test.csv
  results/qwen_qlora_k1_seed0_val.csv
  results/qwen_qlora_k1_seed1_test.csv
  results/qwen_qlora_k1_seed1_val.csv
  results/qwen_qlora_k1_seed2_test.csv
  results/qwen_qlora_k1_seed2_val.csv
  results/qwen_qlora_k20_seed0_test.csv
  results/qwen_qlora_k20_seed0_val.csv
  results/qwen_qlora_k20_seed1_test.csv
  results/qwen_qlora_k20_seed1_val.csv
  results/qwen_qlora_k20_seed2_test.csv
  results/qwen_qlora_k20_seed2_val.csv
  results/qwen_qlora_k30_seed0_test.csv
  results/qwen_qlora_k30_seed0_val.csv
  results/qwen_qlora_k30_seed1_test.csv
  results/qwen_qlora_k30_seed1_val.csv
  results/qwen_qlora_k30_seed2_test.csv
  results/qwen_qlora_k30_seed2_val.csv
  results/qwen_qlora_k3_seed0_t

In [30]:
!git config user.email "shuchivardan@gmail.com"
!git config user.name "Shuchi Vardan"
!git add results/
!git commit -m "Add Qwen SFT test predictions for all budgets and seeds"
!git push origin shuchi/dev

[shuchi/dev d705d3d] Add Qwen SFT test predictions for all budgets and seeds
 21 files changed, 64703 insertions(+)
 create mode 100644 results/qwen_qlora_k10_seed0_test.csv
 create mode 100644 results/qwen_qlora_k10_seed1_test.csv
 create mode 100644 results/qwen_qlora_k10_seed2_test.csv
 create mode 100644 results/qwen_qlora_k1_seed0_test.csv
 create mode 100644 results/qwen_qlora_k1_seed1_test.csv
 create mode 100644 results/qwen_qlora_k1_seed2_test.csv
 create mode 100644 results/qwen_qlora_k20_seed0_test.csv
 create mode 100644 results/qwen_qlora_k20_seed1_test.csv
 create mode 100644 results/qwen_qlora_k20_seed2_test.csv
 create mode 100644 results/qwen_qlora_k30_seed0_test.csv
 create mode 100644 results/qwen_qlora_k30_seed1_test.csv
 create mode 100644 results/qwen_qlora_k30_seed2_test.csv
 create mode 100644 results/qwen_qlora_k3_seed0_test.csv
 create mode 100644 results/qwen_qlora_k3_seed1_test.csv
 create mode 100644 results/qwen_qlora_k3_seed2_test.csv
 create mode 100644 